In [1]:
class DGIM:
    def __init__(self, window_size):
        self.N = window_size
        self.buckets = []
        self.time = 0

    def add_bit(self, bit):
        self.time += 1

        if bit == 1:
            self.buckets.insert(0, [1, self.time])

        # Remove expired buckets
        self.buckets = [
            b for b in self.buckets
            if b[1] > self.time - self.N
        ]

        # Merge buckets when 3 of the same size exist
        while True:
            merged = False
            sizes = set(b[0] for b in self.buckets)

            for size in sorted(sizes):
                indices = [
                    i for i, b in enumerate(self.buckets)
                    if b[0] == size
                ]

                if len(indices) >= 3:
                    # Merge the two oldest buckets
                    i1, i2 = indices[-2], indices[-1]

                    new_bucket = [
                        size * 2,
                        self.buckets[i1][1]
                    ]

                    self.buckets.pop(i2)
                    self.buckets.pop(i1)
                    self.buckets.append(new_bucket)

                    self.buckets.sort(
                        key=lambda x: x[1],
                        reverse=True
                    )

                    merged = True
                    break

            if not merged:
                break

    def estimate(self, k):
        if k < 1 or k > self.N:
            raise ValueError("k must be between 1 and N")

        boundary = self.time - k
        selected = [
            b for b in self.buckets
            if b[1] > boundary
        ]

        if not selected:
            return 0

        # Count all buckets except the oldest
        count = sum(b[0] for b in selected[:-1])

        oldest = selected[-1]
        start_time = oldest[1] - oldest[0] + 1

        # Count the oldest bucket fully or approximately half
        if start_time > boundary:
            count += oldest[0]
        else:
            count += oldest[0] / 2

        return count


# Main Program
window_size = 10

dgim = DGIM(window_size)

# Input binary stream
stream = "1011011010110101"

print("Binary Stream:", stream)

for bit in stream:
    dgim.add_bit(int(bit))

print("\nBuckets (Size, End Time):")
print(dgim.buckets)

k = 10

print("\nLast", k, "bits:")
print("Estimated number of 1s:", dgim.estimate(k))

# Actual count for comparison
actual = stream[-k:].count('1')
print("Actual number of 1s:", actual)


Binary Stream: 1011011010110101

Buckets (Size, End Time):
[[1, 16], [1, 14], [2, 12], [2, 9]]

Last 10 bits:
Estimated number of 1s: 6
Actual number of 1s: 6
